In [ ]:
from glob import glob

docs = glob("../data/raw/doc2phrases_dataset/**")
pdfs = glob("../data/raw/pdf2phrases_dataset/**")

print(f"docs: {len(docs)}, pdfs: {len(pdfs)}")

In [ ]:
def read_text(path):
    with open(path) as text:
        text = text.readlines()

    return text


docs_texts = [
    {"filename": path.split("/")[-1], "text": t}
    for path in docs
    for t in read_text(path)
]

pdfs_texts = [
    {"filename": path.split("/")[-1], "text": t}
    for path in pdfs
    for t in read_text(path)
]

print(f"docs sentences: {len(docs_texts)}, pdfs sentences: {len(pdfs_texts)}")

In [ ]:
# Remove newlines
for d in docs_texts:
    d["text"] = d["text"].replace("\n", "")
for d in pdfs_texts:
    d["text"] = d["text"].replace("\n", "")

In [ ]:
from transformers import pipeline

model_name = "hseBert-it-cased"

pl = pipeline(
    "ner",
    model=f"../models/{model_name.split('/')[-1]}/best_model",
    aggregation_strategy="first",
)

In [ ]:
from tqdm.notebook import tqdm

output = []
i = 0
for sent in tqdm(docs_texts + pdfs_texts):
    pred = pl(sent["text"])
    if not pred:
        continue

    labels = []
    for p in pred:
        if p["score"] > 0.70 and len(p["word"]) > 1:  # type:ignore
            labels.append((p["start"], p["end"], p["entity_group"], float(p["score"])))  # type:ignore

    out = {
        "id": i,
        "text": sent["text"],
        "filename": sent["filename"],
        "label": labels,
    }

    if len(labels) > 0:
        output.append(out)
        i += 1

In [ ]:
import json

with open("../data/labelled/whole_pdf_docs.jsonl", "w") as outfile:
    for obj in output:
        json.dump(obj, outfile)
        outfile.write("\n")  # Add a newline character to separate JSON objects

In [ ]:
import pandas as pd

df = pd.read_json("../data/labelled/whole_pdf_docs.jsonl", lines=True)